# EIS — Train surge + fare models on Colab

This notebook trains both on-device models for the **Unified Urban Mobility & Surge Pricing Predictor** Android app:

1. **Surge LSTM forecaster** — predicts the next 30 min of price movement from the last 2 h of history.
2. **Fare estimator MLP** — predicts the actual rupee fare for a (distance, duration, provider, tier) tuple under current traffic.

It produces four files in `/content/`:

```
surge_forecast.tflite
surge_forecast_meta.json
fare_estimator.tflite
fare_estimator_meta.json
```

The last cell downloads them to your machine — drop all four into `app/src/main/assets/` in the Android project and rebuild.

**Runtime:** ~2–3 minutes on a Colab CPU. No GPU needed.

## 1. Verify environment

Colab already ships TensorFlow + NumPy. Pinning older versions causes a numpy ABI mismatch on import (`numpy.dtype size changed`) because the pre-installed wheels were compiled against the version Colab shipped — so we don't pin anything.

If a prior run downgraded numpy and you hit that error, recover with **Runtime → Restart session** and re-run from the top.

In [ ]:
import tensorflow as tf, numpy as np
print(f'tensorflow {tf.__version__}  numpy {np.__version__}')

## 2. Upload the trainer

Option A (recommended) — upload `train_on_colab.py` from the repo's `ml_model/colab/` folder when prompted.

Option B — skip this cell and paste the contents of that file inline into a code cell instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick train_on_colab.py from the file chooser
assert 'train_on_colab.py' in uploaded, 'Please upload train_on_colab.py'

## 3. Train both models

This runs the surge LSTM first (~1 min) then the fare MLP (~1 min). Progress prints inline.

In [ ]:
%run train_on_colab.py

## 4. Sanity-check the artefacts

Confirms all four files are on disk and prints their sizes + the normalisation constants baked into the meta JSONs. If a file is missing, re-run cell 3.

In [ ]:
import json, pathlib

artefacts = [
    'surge_forecast.tflite',
    'surge_forecast_meta.json',
    'fare_estimator.tflite',
    'fare_estimator_meta.json',
]
for name in artefacts:
    p = pathlib.Path(name)
    assert p.exists(), f'MISSING: {name}'
    print(f'{name:32s} {p.stat().st_size / 1024:>8.1f} KiB')

print('\nSurge meta:')
print(json.dumps(json.loads(pathlib.Path('surge_forecast_meta.json').read_text()), indent=2))
print('\nFare meta:')
print(json.dumps(json.loads(pathlib.Path('fare_estimator_meta.json').read_text()), indent=2))

## 5. Download to your machine

Browser triggers four download prompts. Save each file straight into `app/src/main/assets/` in the Android project, then rebuild.

In [ ]:
from google.colab import files
for name in [
    'surge_forecast.tflite',
    'surge_forecast_meta.json',
    'fare_estimator.tflite',
    'fare_estimator_meta.json',
]:
    files.download(name)